# Privileged Access Management + Cloud Resource Exposure

## Executive Summary

This notebook demonstrates Microsoft Sentinel Custom Graph capabilities for 3rd party ISVs by showcasing how ISV data (Privileged Access Management) can be combined with Microsoft security data to perform advanced graph analytics similar to Microsoft Security Exposure Management (MSEM).

* **Scenario:** Cloud resource exposure through misconfigured PAM sessions
* **Approach:** Progressive attack narrative using all 5 built-in algorithms + GraphFrames extensibility
* **Data Scale:** ≤50 rows per table, 100% synthetic data with realistic relationships

## Goal

Convey the features, advantages, and benefits of Microsoft Sentinel Custom Graphs to Microsoft's 3rd party Independent Software Vendors (ISVs) by demonstrating how they can:

1. Integrate their security data with Microsoft's native security telemetry
2. Perform advanced graph analysis (blast radius, attack paths, centrality)
3. Extend capabilities with GraphFrames for custom algorithms
4. Gain insights similar to MSEM but with their own proprietary data

## Narrative Arc

1. **Discovery (Centrality)**: Security team identifies which PAM vaults and cloud resources are most critical based on connection patterns.
2. **Discovery (K-Hop)**: A contractor account is compromised. Discover everything they can reach within 3 hops to understand attack surface.
3. **Access Control Validation (Reachability)**: Check if compromised account can reach production databases through PAM sessions (policy violation detection).
4. **Impact Assessment (Blast Radius)**: Measure ALL possible paths from compromised contractor to critical production SQL server.
5. **Attack Path Analysis (Ranked Paths)**: Rank attack paths by combined risk score to prioritize which paths to remediate first (MSEM "Attack Path").
6. **Advanced Analysis (GraphFrames)**: Use PageRank to identify most influential accounts/resources for monitoring.

## References

### Microsoft Documentation
- [Blog: Announcing Custom Graphs Public Preview](https://techcommunity.microsoft.com/blog/microsoft-security-blog/announcing-public-preview-of-custom-graphs-in-microsoft-sentinel/4507410)
- [Blog: Exposure Management Graph](https://techcommunity.microsoft.com/blog/microsoft-security-blog/microsoft-security-exposure-management-graph-unveiling-the-power/4148546)
- [Available Defender Tables](https://learn.microsoft.com/en-us/defender-xdr/advanced-hunting-schema-tables)
- [Available Azure Monitor Tables](https://learn.microsoft.com/en-us/azure/azure-monitor/reference/tables-index)
- [Available Asset Tables](https://learn.microsoft.com/en-us/azure/sentinel/datalake/enable-data-connectors)
- [Microsoft Sentinel Notebook Examples](https://learn.microsoft.com/en-us/azure/sentinel/datalake/notebook-examples)
- [Microsoft Sentinel Provider Class Reference](https://learn.microsoft.com/en-us/azure/sentinel/datalake/sentinel-provider-class-reference)
- [Custom graph overview doc](https://learn.microsoft.com/en-us/azure/sentinel/datalake/custom-graphs-overview)
- [Graph Query Language (GQL) Reference for Sentinel graph](https://learn.microsoft.com/en-us/azure/sentinel/datalake/gql-reference-for-sentinel-custom-graph)
- [Graph REST APIs for custom graphs](https://learn.microsoft.com/en-us/azure/sentinel/datalake/graph-rest-api)

### External Resources
- [Neo4j Graph Data Science Documentation](https://neo4j.com/docs/graph-data-science/current/)
- [GraphFrames Documentation](https://graphframes.github.io/graphframes/docs/_site/)
- [NetworkX Algorithms Reference](https://networkx.org/documentation/stable/reference/algorithms/)



## Section 1: Introduction & Setup
- Import libraries
- Configure Sentinel provider
- Explain scenario and objectives

In [ ]:
# Import required libraries
# Required packages
from sentinel_lake.providers import MicrosoftSentinelProvider
from sentinel_graph.builders import GraphSpecBuilder
from sentinel_graph.builders.query_input import (
    CentralityQueryInput,
    K_HopQueryInput,
    BlastRadiusQueryInput,
    ReachabilityQueryInput,
    RankedQueryInput
)
from pyspark.sql.functions import col, lit, concat
from pyspark.sql.functions import (
    col, count, countDistinct, when, lit, expr,
    current_timestamp, avg, coalesce
)
import matplotlib.pyplot as plt
import pandas as pd

# Configuration - Update this with your workspace name
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"

# Configuration - Whether we are generating demo data (true) or using real data (false)
DEMO_MODE = True

# Configuration - Lookback period for Analysis
ANALYSIS_DAYS = 14

# Initialize Sentinel provider
sentinel_provider = MicrosoftSentinelProvider(spark)

print("="*60)
print("SCENARIO: Privileged Access Management + Cloud Resource Exposure")
print("="*60)
print(f"Analysis Window: {ANALYSIS_DAYS} days")
print(f"Demo Mode: {DEMO_MODE}")
print(f"Workspace: {WORKSPACE_NAME}")
print("="*60)

## Section 2: Data Generation
- Create synthetic DataFrames for all tables
- Show sample data for transparency
- Explain relationships

### Microsoft Data (3 tables)


In [ ]:
# Section 2: Data Generation - Microsoft Tables
# Create synthetic DataFrames matching Microsoft Sentinel table schemas

from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, BooleanType
from datetime import datetime, timedelta
import random

# Helper function to generate timestamps within analysis window
def generate_timestamp(days_ago=0):
    """Generate timestamp within analysis window"""
    base = datetime.now() - timedelta(days=days_ago)
    return base.strftime("%Y-%m-%d %H:%M:%S")

# ============================================================================
# 1. EntraUsers Table (30 users)
# ============================================================================
# Properties: id, displayName, department, userType, accountEnabled

entra_users_data = [
    # Regular Users (20)
    ("user-001", "Sarah Johnson", "Engineering", "Member", True),
    ("user-002", "Michael Chen", "Engineering", "Member", True),
    ("user-003", "Emma Williams", "Engineering", "Member", True),
    ("user-004", "David Martinez", "Engineering", "Member", True),
    ("user-005", "Lisa Anderson", "IT Operations", "Member", True),
    ("user-006", "James Taylor", "IT Operations", "Member", True),
    ("user-007", "Maria Garcia", "IT Operations", "Member", True),
    ("user-008", "Robert Brown", "Security", "Member", True),
    ("user-009", "Jennifer Davis", "Security", "Member", True),
    ("user-010", "William Miller", "Finance", "Member", True),
    ("user-011", "Patricia Wilson", "Finance", "Member", True),
    ("user-012", "Richard Moore", "HR", "Member", True),
    ("user-013", "Linda Jackson", "HR", "Member", True),
    ("user-014", "Christopher White", "Marketing", "Member", True),
    ("user-015", "Barbara Harris", "Marketing", "Member", True),
    ("user-016", "Daniel Martin", "Engineering", "Member", True),
    ("user-017", "Susan Thompson", "Engineering", "Member", True),
    ("user-018", "Matthew Garcia", "IT Operations", "Member", True),
    ("user-019", "Nancy Robinson", "Security", "Member", True),
    ("user-020", "Joseph Clark", "Finance", "Member", True),
    
    # Contractors (5) - Including our compromised contractor
    ("user-contractor-alex-chen", "Alex Chen", "Engineering", "Contractor", True),
    ("user-contractor-sam-rivera", "Sam Rivera", "IT Operations", "Contractor", True),
    ("user-contractor-morgan-lee", "Morgan Lee", "Engineering", "Contractor", True),
    ("user-contractor-casey-kim", "Casey Kim", "Security", "Contractor", True),
    ("user-contractor-taylor-patel", "Taylor Patel", "IT Operations", "Contractor", True),
    
    # Admins (5)
    ("user-admin-jordan-park", "Jordan Park", "IT Operations", "Admin", True),
    ("user-admin-alex-nguyen", "Alex Nguyen", "Security", "Admin", True),
    ("user-admin-sam-rodriguez", "Sam Rodriguez", "IT Operations", "Admin", True),
    ("user-admin-chris-lewis", "Chris Lewis", "Security", "Admin", True),
    ("user-admin-pat-walker", "Pat Walker", "Engineering", "Admin", True),
]

entra_users_schema = StructType([
    StructField("id", StringType(), False),
    StructField("displayName", StringType(), True),
    StructField("department", StringType(), True),
    StructField("userType", StringType(), True),
    StructField("accountEnabled", BooleanType(), True),
])

entra_users_df = spark.createDataFrame(entra_users_data, entra_users_schema)

print(f"✅ Created EntraUsers table: {entra_users_df.count()} users")
print("\nSample EntraUsers:")
entra_users_df.show(5, truncate=False)

# ============================================================================
# 2. AzureActivity Table (25 resources)
# ============================================================================
# Properties: ResourceId, ResourceGroup, ResourceProvider, ResourceType, SubscriptionId

azure_resources_data = [
    # Production SQL Databases (3) - Critical assets
    ("resource-prod-sql-01", "Production", "Microsoft.Sql", "SQL Database", "sub-prod-001"),
    ("resource-prod-sql-02", "Production", "Microsoft.Sql", "SQL Database", "sub-prod-001"),
    ("resource-prod-sql-backup", "Production", "Microsoft.Sql", "SQL Database", "sub-prod-002"),
    
    # Production VMs (5)
    ("resource-prod-vm-web-01", "Production", "Microsoft.Compute", "Virtual Machine", "sub-prod-001"),
    ("resource-prod-vm-web-02", "Production", "Microsoft.Compute", "Virtual Machine", "sub-prod-001"),
    ("resource-prod-vm-api-01", "Production", "Microsoft.Compute", "Virtual Machine", "sub-prod-001"),
    ("resource-prod-vm-api-02", "Production", "Microsoft.Compute", "Virtual Machine", "sub-prod-001"),
    ("resource-prod-vm-backup-01", "Production", "Microsoft.Compute", "Virtual Machine", "sub-prod-001"),
    
    # Development VMs (3)
    ("resource-dev-vm-01", "Development", "Microsoft.Compute", "Virtual Machine", "sub-dev-001"),
    ("resource-dev-vm-02", "Development", "Microsoft.Compute", "Virtual Machine", "sub-dev-001"),
    ("resource-dev-vm-03", "Development", "Microsoft.Compute", "Virtual Machine", "sub-dev-001"),
    
    # Test VMs (3)
    ("resource-test-vm-01", "Test", "Microsoft.Compute", "Virtual Machine", "sub-test-001"),
    ("resource-test-vm-02", "Test", "Microsoft.Compute", "Virtual Machine", "sub-test-001"),
    ("resource-test-vm-03", "Test", "Microsoft.Compute", "Virtual Machine", "sub-test-001"),
    
    # Storage Accounts (7)
    ("resource-prod-storage-01", "Production", "Microsoft.Storage", "Storage Account", "sub-prod-001"),
    ("resource-prod-storage-02", "Production", "Microsoft.Storage", "Storage Account", "sub-prod-002"),
    ("resource-dev-storage-01", "Development", "Microsoft.Storage", "Storage Account", "sub-dev-001"),
    ("resource-dev-storage-02", "Development", "Microsoft.Storage", "Storage Account", "sub-dev-001"),
    ("resource-test-storage-01", "Test", "Microsoft.Storage", "Storage Account", "sub-test-001"),
    ("resource-backup-storage-01", "Production", "Microsoft.Storage", "Storage Account", "sub-prod-002"),
    ("resource-logs-storage-01", "Production", "Microsoft.Storage", "Storage Account", "sub-prod-001"),
    
    # Key Vaults (5) - Sensitive credentials
    ("resource-prod-keyvault-01", "Production", "Microsoft.KeyVault", "Key Vault", "sub-prod-001"),
    ("resource-prod-keyvault-02", "Production", "Microsoft.KeyVault", "Key Vault", "sub-prod-002"),
    ("resource-dev-keyvault-01", "Development", "Microsoft.KeyVault", "Key Vault", "sub-dev-001"),
    ("resource-test-keyvault-01", "Test", "Microsoft.KeyVault", "Key Vault", "sub-test-001"),
    ("resource-admin-keyvault-01", "Production", "Microsoft.KeyVault", "Key Vault", "sub-prod-001"),
]

azure_resources_schema = StructType([
    StructField("ResourceId", StringType(), False),
    StructField("ResourceGroup", StringType(), True),
    StructField("ResourceProvider", StringType(), True),
    StructField("ResourceType", StringType(), True),
    StructField("SubscriptionId", StringType(), True),
])

azure_activity_df = spark.createDataFrame(azure_resources_data, azure_resources_schema)

print(f"\n✅ Created AzureActivity table: {azure_activity_df.count()} resources")
print("\nSample AzureActivity:")
azure_activity_df.show(5, truncate=False)

# ============================================================================
# 3. AuditLogs Table - Service Principals (15)
# ============================================================================
# Properties: AppId, ServicePrincipalName, displayName

service_principals_data = [
    # High-privilege automation accounts
    ("app-datasyncbot", "DataSyncBot", "DataSyncBot"),  # Over-privileged (8 resources)
    ("app-backupservice", "BackupService", "Backup Service"),
    ("app-monitoragent", "MonitoringAgent", "Azure Monitor Agent"),
    
    # Application service principals
    ("app-webapp-prod-01", "WebApp-Prod-01", "Production Web Application"),
    ("app-webapp-prod-02", "WebApp-Prod-02", "Production Web Application"),
    ("app-api-prod-01", "API-Prod-01", "Production API Service"),
    ("app-api-prod-02", "API-Prod-02", "Production API Service"),
    
    # Development service principals
    ("app-webapp-dev-01", "WebApp-Dev-01", "Development Web Application"),
    ("app-api-dev-01", "API-Dev-01", "Development API Service"),
    ("app-testrunner-01", "TestRunner-01", "Automated Test Runner"),
    
    # Infrastructure automation
    ("app-terraform", "Terraform-Automation", "Terraform Infrastructure Automation"),
    ("app-ansible", "Ansible-Config", "Ansible Configuration Management"),
    ("app-cicd-pipeline", "CICD-Pipeline", "CI/CD Deployment Pipeline"),
    
    # Security tools
    ("app-vulnerability-scanner", "VulnScanner", "Vulnerability Scanner"),
    ("app-compliance-checker", "ComplianceChecker", "Compliance Checker"),
]

service_principals_schema = StructType([
    StructField("AppId", StringType(), False),
    StructField("ServicePrincipalName", StringType(), True),
    StructField("displayName", StringType(), True),
])

audit_logs_df = spark.createDataFrame(service_principals_data, service_principals_schema)

print(f"\n✅ Created AuditLogs (ServicePrincipals) table: {audit_logs_df.count()} service principals")
print("\nSample ServicePrincipals:")
audit_logs_df.show(5, truncate=False)

# ============================================================================
# Summary Statistics
# ============================================================================
print("\n" + "="*60)
print("MICROSOFT DATA SUMMARY")
print("="*60)
print(f"EntraUsers:           {entra_users_df.count()} users")
print(f"  - Regular Users:    20")
print(f"  - Contractors:      5 (including Alex Chen - compromised)")
print(f"  - Admins:           5 (including Jordan Park - high privilege)")
print(f"\nAzureActivity:        {azure_activity_df.count()} resources")
print(f"  - SQL Databases:    3 (Production)")
print(f"  - Virtual Machines: 10 (4 prod, 3 dev, 3 test)")
print(f"  - Storage Accounts: 7 (various environments)")
print(f"  - Key Vaults:       5 (sensitive credentials)")
print(f"\nServicePrincipals:    {audit_logs_df.count()} apps")
print(f"  - DataSyncBot:      Over-privileged (8 resources)")
print(f"  - Infrastructure:   Terraform, Ansible, CI/CD")
print(f"  - Security Tools:   Vuln Scanner, Compliance")
print("="*60)


### ISV Data - PAM Vendor (2 node types)

In [ ]:
# Section 2: ISV Data Generation - PAM Tables
# Create synthetic PAM vendor data (Privileged Access Management)

from datetime import datetime, timedelta

# ============================================================================
# 4. PAMVault Table (5 vaults)
# ============================================================================
# Properties: vault_id, vault_name, sensitivity_level, compliance_tier

pam_vaults_data = [
    ("vault-prod", "Production Vault", "Critical", "Tier-1"),
    ("vault-dev", "Development Vault", "Medium", "Tier-2"),
    ("vault-admin", "Admin Vault", "Critical", "Tier-1"),
    ("vault-contractor", "Contractor Vault", "Medium", "Tier-2"),  # Misconfigured!
    ("vault-app", "Application Vault", "High", "Tier-1"),
]

pam_vaults_schema = StructType([
    StructField("vault_id", StringType(), False),
    StructField("vault_name", StringType(), True),
    StructField("sensitivity_level", StringType(), True),
    StructField("compliance_tier", StringType(), True),
])

pam_vaults_df = spark.createDataFrame(pam_vaults_data, pam_vaults_schema)
print(f"✅ Created PAMVaults: {pam_vaults_df.count()} vaults")
pam_vaults_df.show(truncate=False)

# ============================================================================
# 5. PAMSession Table (40 sessions)
# ============================================================================
# Properties: session_id, user_id, vault_id, start_time, duration_minutes, 
#             elevation_reason, approval_status

# Helper to generate recent timestamps
base_time = datetime.now()

pam_sessions_data = [
    # Alex Chen (Contractor) sessions - Some problematic
    ("session-001", "user-contractor-alex-chen", "vault-contractor", (base_time - timedelta(days=1)).isoformat(), 45, "Development work", "Approved"),
    ("session-023", "user-contractor-alex-chen", "vault-contractor", (base_time - timedelta(days=2)).isoformat(), 120, "Database migration", "Approved"),  # Access to prod SQL!
    ("session-031", "user-contractor-sam-rivera", "vault-contractor", (base_time - timedelta(days=3)).isoformat(), 90, "Configuration update", "Approved"),
    ("session-050", "user-contractor-alex-chen", "vault-contractor", (base_time - timedelta(days=1)).isoformat(), 45, "Backup maintenance", "Approved"),
    
    # Regular user sessions
    ("session-002", "user-001", "vault-dev", (base_time - timedelta(days=1, hours=2)).isoformat(), 30, "Testing", "Auto-approved"),
    ("session-003", "user-002", "vault-dev", (base_time - timedelta(days=1, hours=4)).isoformat(), 60, "Development", "Auto-approved"),
    ("session-004", "user-003", "vault-dev", (base_time - timedelta(days=2, hours=1)).isoformat(), 45, "Testing", "Auto-approved"),
    ("session-005", "user-004", "vault-dev", (base_time - timedelta(days=2, hours=3)).isoformat(), 75, "Development", "Auto-approved"),
    ("session-006", "user-005", "vault-dev", (base_time - timedelta(days=3, hours=2)).isoformat(), 30, "Testing", "Auto-approved"),
    ("session-007", "user-006", "vault-app", (base_time - timedelta(days=1, hours=5)).isoformat(), 90, "App deployment", "Approved"),
    ("session-008", "user-007", "vault-app", (base_time - timedelta(days=2, hours=2)).isoformat(), 60, "App config", "Approved"),
    ("session-009", "user-008", "vault-prod", (base_time - timedelta(days=1, hours=3)).isoformat(), 120, "Security audit", "Approved"),
    ("session-010", "user-009", "vault-prod", (base_time - timedelta(days=2, hours=4)).isoformat(), 90, "Incident response", "Approved"),
    
    # Admin sessions - Jordan Park (high privilege)
    ("session-011", "user-admin-jordan-park", "vault-admin", (base_time - timedelta(days=1, hours=1)).isoformat(), 180, "System maintenance", "Approved"),
    ("session-012", "user-admin-jordan-park", "vault-prod", (base_time - timedelta(days=2, hours=2)).isoformat(), 150, "Production deployment", "Approved"),
    ("session-013", "user-admin-jordan-park", "vault-app", (base_time - timedelta(days=3, hours=1)).isoformat(), 120, "Service principal rotation", "Approved"),
    ("session-014", "user-admin-alex-nguyen", "vault-admin", (base_time - timedelta(days=1, hours=6)).isoformat(), 90, "Security review", "Approved"),
    ("session-015", "user-admin-sam-rodriguez", "vault-prod", (base_time - timedelta(days=2, hours=5)).isoformat(), 120, "Backup verification", "Approved"),
    
    # More regular user sessions
    ("session-016", "user-010", "vault-dev", (base_time - timedelta(days=4, hours=1)).isoformat(), 45, "Testing", "Auto-approved"),
    ("session-017", "user-011", "vault-dev", (base_time - timedelta(days=4, hours=3)).isoformat(), 60, "Development", "Auto-approved"),
    ("session-018", "user-012", "vault-app", (base_time - timedelta(days=3, hours=4)).isoformat(), 75, "HR system access", "Approved"),
    ("session-019", "user-013", "vault-app", (base_time - timedelta(days=3, hours=5)).isoformat(), 60, "HR reports", "Approved"),
    ("session-020", "user-014", "vault-dev", (base_time - timedelta(days=5, hours=2)).isoformat(), 45, "Marketing dashboard", "Auto-approved"),
    
    # Contractor sessions (additional)
    ("session-021", "user-contractor-morgan-lee", "vault-contractor", (base_time - timedelta(days=1, hours=7)).isoformat(), 60, "Code review", "Approved"),
    ("session-022", "user-contractor-casey-kim", "vault-contractor", (base_time - timedelta(days=2, hours=6)).isoformat(), 90, "Security testing", "Approved"),
    
    # Service principal rotation sessions
    ("session-024", "user-admin-jordan-park", "vault-app", (base_time - timedelta(days=4, hours=2)).isoformat(), 30, "SP credential rotation", "Approved"),
    ("session-025", "user-admin-pat-walker", "vault-app", (base_time - timedelta(days=5, hours=1)).isoformat(), 45, "SP setup", "Approved"),
    
    # Production access sessions
    ("session-026", "user-005", "vault-prod", (base_time - timedelta(days=3, hours=3)).isoformat(), 120, "Production support", "Approved"),
    ("session-027", "user-006", "vault-prod", (base_time - timedelta(days=4, hours=4)).isoformat(), 90, "Production monitoring", "Approved"),
    ("session-028", "user-008", "vault-prod", (base_time - timedelta(days=5, hours=3)).isoformat(), 150, "Security incident", "Emergency-approved"),
    
    # Development sessions
    ("session-029", "user-016", "vault-dev", (base_time - timedelta(days=6, hours=1)).isoformat(), 60, "Feature development", "Auto-approved"),
    ("session-030", "user-017", "vault-dev", (base_time - timedelta(days=6, hours=2)).isoformat(), 45, "Bug fixing", "Auto-approved"),
    
    # Additional contractor and admin sessions
    ("session-032", "user-contractor-taylor-patel", "vault-contractor", (base_time - timedelta(days=4, hours=5)).isoformat(), 75, "Infrastructure work", "Approved"),
    ("session-033", "user-admin-chris-lewis", "vault-admin", (base_time - timedelta(days=5, hours=4)).isoformat(), 120, "Admin tasks", "Approved"),
    ("session-034", "user-018", "vault-app", (base_time - timedelta(days=6, hours=3)).isoformat(), 60, "Application support", "Approved"),
    ("session-035", "user-019", "vault-prod", (base_time - timedelta(days=7, hours=2)).isoformat(), 90, "Security audit", "Approved"),
    ("session-036", "user-020", "vault-dev", (base_time - timedelta(days=7, hours=1)).isoformat(), 45, "Financial reports", "Auto-approved"),
    
    # Recent sessions
    ("session-037", "user-001", "vault-dev", (base_time - timedelta(hours=8)).isoformat(), 60, "Daily development", "Auto-approved"),
    ("session-038", "user-admin-jordan-park", "vault-prod", (base_time - timedelta(hours=12)).isoformat(), 90, "Production check", "Approved"),
    ("session-039", "user-contractor-alex-chen", "vault-contractor", (base_time - timedelta(hours=16)).isoformat(), 30, "Quick fix", "Approved"),
    ("session-040", "user-009", "vault-admin", (base_time - timedelta(hours=20)).isoformat(), 75, "Security review", "Approved"),
]

pam_sessions_schema = StructType([
    StructField("session_id", StringType(), False),
    StructField("user_id", StringType(), True),
    StructField("vault_id", StringType(), True),
    StructField("start_time", StringType(), True),
    StructField("duration_minutes", IntegerType(), True),
    StructField("elevation_reason", StringType(), True),
    StructField("approval_status", StringType(), True),
])

pam_sessions_df = spark.createDataFrame(pam_sessions_data, pam_sessions_schema)
print(f"\n✅ Created PAMSessions: {pam_sessions_df.count()} sessions")
print("\nSample PAMSessions (including problematic sessions):")
pam_sessions_df.filter(col("user_id").like("%contractor%")).show(5, truncate=False)

# Summary
print("\n" + "="*60)
print("ISV PAM DATA SUMMARY")
print("="*60)
print(f"PAM Vaults:        {pam_vaults_df.count()} vaults")
print(f"  - Production:    1 (Critical/Tier-1)")
print(f"  - Admin:         1 (Critical/Tier-1)")
print(f"  - Contractor:    1 (Medium/Tier-2) ⚠️ MISCONFIGURED")
print(f"  - Development:   1 (Medium/Tier-2)")
print(f"  - Application:   1 (High/Tier-1)")
print(f"\nPAM Sessions:      {pam_sessions_df.count()} sessions")
print(f"  - Contractor:    7 sessions (Alex Chen, Sam Rivera, etc.)")
print(f"  - Admin:         8 sessions (Jordan Park - 4 sessions)")
print(f"  - Regular Users: 25 sessions")
print(f"\n⚠️ KEY FINDINGS:")
print(f"  - Session #023: Alex Chen → Contractor Vault → Prod SQL")
print(f"  - Session #031: Sam Rivera → Contractor Vault → Prod KeyVault")
print(f"  - Jordan Park has access to 4 different vaults (high privilege)")
print("="*60)


### Edge data frames

In [ ]:
# ============================================================================
# Edge DataFrames (6 edge types) - Create relationships between nodes
# ============================================================================

# Edge 1: ACCESSED (User → Resource) - 45 edges
accessed_edges_data = [
    # Contractors accessing dev resources (legitimate)
    ("user-contractor-alex-chen", "resource-dev-vm-01", "Deploy", "2026-01-12 09:00:00", "Success"),
    ("user-contractor-alex-chen", "resource-dev-storage-01", "Read", "2026-01-12 10:00:00", "Success"),
    ("user-contractor-sam-rivera", "resource-dev-vm-02", "Deploy", "2026-01-11 09:00:00", "Success"),
    ("user-contractor-morgan-lee", "resource-dev-vm-03", "Configure", "2026-01-10 14:00:00", "Success"),
    ("user-contractor-casey-kim", "resource-test-vm-01", "Test", "2026-01-10 11:00:00", "Success"),
    
    # Regular engineering users
    ("user-001", "resource-dev-vm-01", "Deploy", "2026-01-13 09:00:00", "Success"),
    ("user-001", "resource-dev-storage-01", "Read", "2026-01-13 10:00:00", "Success"),
    ("user-002", "resource-dev-vm-02", "Configure", "2026-01-13 11:00:00", "Success"),
    ("user-003", "resource-dev-storage-02", "Write", "2026-01-12 14:00:00", "Success"),
    ("user-004", "resource-dev-vm-03", "Deploy", "2026-01-12 15:00:00", "Success"),
    ("user-016", "resource-dev-vm-01", "Debug", "2026-01-11 09:00:00", "Success"),
    ("user-017", "resource-dev-storage-01", "Read", "2026-01-11 10:00:00", "Success"),
    
    # IT Ops users
    ("user-005", "resource-dev-vm-02", "Monitor", "2026-01-13 08:00:00", "Success"),
    ("user-006", "resource-prod-vm-web-01", "Monitor", "2026-01-13 09:00:00", "Success"),
    ("user-007", "resource-prod-storage-01", "Monitor", "2026-01-12 08:00:00", "Success"),
    ("user-018", "resource-dev-vm-03", "Configure", "2026-01-11 13:00:00", "Success"),
    
    # Security users
    ("user-008", "resource-prod-sql-01", "Audit", "2026-01-13 10:00:00", "Success"),
    ("user-009", "resource-prod-keyvault-01", "Audit", "2026-01-12 11:00:00", "Success"),
    ("user-019", "resource-admin-keyvault-01", "Review", "2026-01-11 14:00:00", "Success"),
    
    # Finance users
    ("user-010", "resource-prod-sql-01", "Report", "2026-01-13 13:00:00", "Success"),
    ("user-011", "resource-prod-storage-02", "Report", "2026-01-12 13:00:00", "Success"),
    ("user-020", "resource-dev-storage-01", "Test", "2026-01-11 12:00:00", "Success"),
    
    # Admins accessing production (legitimate admin access)
    ("user-admin-jordan-park", "resource-prod-vm-web-01", "Restart", "2026-01-14 09:00:00", "Success"),
    ("user-admin-jordan-park", "resource-prod-storage-01", "Configure", "2026-01-14 10:00:00", "Success"),
    ("user-admin-jordan-park", "resource-prod-sql-01", "Backup", "2026-01-13 15:00:00", "Success"),
    ("user-admin-jordan-park", "resource-admin-keyvault-01", "Admin", "2026-01-13 16:00:00", "Success"),
    ("user-admin-alex-nguyen", "resource-prod-keyvault-01", "Audit", "2026-01-13 11:00:00", "Success"),
    ("user-admin-alex-nguyen", "resource-admin-keyvault-01", "Configure", "2026-01-12 16:00:00", "Success"),
    ("user-admin-sam-rodriguez", "resource-prod-vm-api-01", "Deploy", "2026-01-13 14:00:00", "Success"),
    ("user-admin-sam-rodriguez", "resource-prod-storage-02", "Backup", "2026-01-12 15:00:00", "Success"),
    ("user-admin-chris-lewis", "resource-prod-sql-02", "Audit", "2026-01-12 12:00:00", "Success"),
    ("user-admin-pat-walker", "resource-prod-vm-web-02", "Deploy", "2026-01-11 15:00:00", "Success"),
    
    # Marketing/HR
    ("user-014", "resource-dev-storage-01", "Read", "2026-01-10 10:00:00", "Success"),
    ("user-015", "resource-dev-storage-02", "Write", "2026-01-10 11:00:00", "Success"),
    ("user-012", "resource-prod-storage-01", "Report", "2026-01-09 14:00:00", "Success"),
    ("user-013", "resource-prod-storage-02", "Report", "2026-01-09 15:00:00", "Success"),
    
    # Test environment access
    ("user-001", "resource-test-vm-01", "Test", "2026-01-10 09:00:00", "Success"),
    ("user-002", "resource-test-vm-02", "Test", "2026-01-10 10:00:00", "Success"),
    ("user-003", "resource-test-storage-01", "Test", "2026-01-09 11:00:00", "Success"),
    ("user-004", "resource-test-keyvault-01", "Test", "2026-01-09 12:00:00", "Success"),
    
    # Additional dev access
    ("user-016", "resource-dev-keyvault-01", "Read", "2026-01-08 09:00:00", "Success"),
    ("user-017", "resource-dev-vm-02", "Deploy", "2026-01-08 10:00:00", "Success"),
    ("user-018", "resource-test-vm-03", "Configure", "2026-01-08 11:00:00", "Success"),
    ("user-005", "resource-test-vm-01", "Monitor", "2026-01-07 14:00:00", "Success"),
    ("user-006", "resource-test-vm-02", "Monitor", "2026-01-07 15:00:00", "Success"),
]

accessed_edges_schema = StructType([
    StructField("source_id", StringType(), False),
    StructField("target_id", StringType(), False),
    StructField("operation", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("result_status", StringType(), True),
])
accessed_edges_df = spark.createDataFrame(accessed_edges_data, accessed_edges_schema)
print(f"✅ Created ACCESSED edges: {accessed_edges_df.count()} relationships")

# Edge 2: OWNS (User → ServicePrincipal) - 15 edges
owns_edges_data = [
    ("user-admin-jordan-park", "app-datasyncbot", "2025-06-15", "Owner"),
    ("user-admin-jordan-park", "app-backupservice", "2025-07-01", "Owner"),
    ("user-admin-alex-nguyen", "app-monitoragent", "2025-06-20", "Owner"),
    ("user-admin-alex-nguyen", "app-vulnerability-scanner", "2025-08-01", "Owner"),
    ("user-admin-sam-rodriguez", "app-terraform", "2025-05-10", "Owner"),
    ("user-admin-sam-rodriguez", "app-ansible", "2025-05-15", "Owner"),
    ("user-admin-chris-lewis", "app-compliance-checker", "2025-07-20", "Owner"),
    ("user-admin-pat-walker", "app-webapp-prod-01", "2025-04-01", "Owner"),
    ("user-admin-pat-walker", "app-webapp-prod-02", "2025-04-01", "Owner"),
    ("user-001", "app-webapp-dev-01", "2025-09-01", "Owner"),
    ("user-002", "app-api-dev-01", "2025-09-05", "Owner"),
    ("user-003", "app-testrunner-01", "2025-09-10", "Owner"),
    ("user-006", "app-cicd-pipeline", "2025-06-01", "Owner"),
    ("user-007", "app-api-prod-01", "2025-05-20", "Owner"),
    ("user-008", "app-api-prod-02", "2025-05-20", "Owner"),
]

owns_edges_schema = StructType([
    StructField("source_id", StringType(), False),
    StructField("target_id", StringType(), False),
    StructField("created_date", StringType(), True),
    StructField("permission_level", StringType(), True),
])
owns_edges_df = spark.createDataFrame(owns_edges_data, owns_edges_schema)
print(f"✅ Created OWNS edges: {owns_edges_df.count()} relationships")

# Edge 3: HAS_PERMISSION (ServicePrincipal → Resource) - 35 edges
has_permission_edges_data = [
    # DataSyncBot - OVER-PRIVILEGED (8 resources)
    ("app-datasyncbot", "resource-prod-sql-01", "ReadWrite", "Subscription", "2025-06-15"),
    ("app-datasyncbot", "resource-prod-sql-02", "ReadWrite", "Subscription", "2025-06-15"),
    ("app-datasyncbot", "resource-prod-storage-01", "ReadWrite", "ResourceGroup", "2025-06-15"),
    ("app-datasyncbot", "resource-prod-storage-02", "ReadWrite", "ResourceGroup", "2025-06-15"),
    ("app-datasyncbot", "resource-dev-storage-01", "ReadWrite", "ResourceGroup", "2025-06-15"),
    ("app-datasyncbot", "resource-dev-storage-02", "ReadWrite", "ResourceGroup", "2025-06-15"),
    ("app-datasyncbot", "resource-prod-keyvault-01", "Get", "Resource", "2025-06-15"),
    ("app-datasyncbot", "resource-dev-keyvault-01", "Get", "Resource", "2025-06-15"),
    
    # BackupService
    ("app-backupservice", "resource-prod-sql-backup", "Backup", "Resource", "2025-07-01"),
    ("app-backupservice", "resource-backup-storage-01", "ReadWrite", "Resource", "2025-07-01"),
    ("app-backupservice", "resource-prod-storage-01", "Read", "Resource", "2025-07-01"),
    
    # MonitorAgent
    ("app-monitoragent", "resource-prod-vm-web-01", "Read", "Subscription", "2025-06-20"),
    ("app-monitoragent", "resource-prod-vm-api-01", "Read", "Subscription", "2025-06-20"),
    ("app-monitoragent", "resource-logs-storage-01", "Write", "ResourceGroup", "2025-06-20"),
    
    # Production apps
    ("app-webapp-prod-01", "resource-prod-sql-01", "ReadWrite", "Resource", "2025-04-01"),
    ("app-webapp-prod-01", "resource-prod-storage-01", "Read", "Resource", "2025-04-01"),
    ("app-webapp-prod-02", "resource-prod-sql-02", "ReadWrite", "Resource", "2025-04-01"),
    ("app-api-prod-01", "resource-prod-sql-01", "Read", "Resource", "2025-05-20"),
    ("app-api-prod-02", "resource-prod-sql-02", "Read", "Resource", "2025-05-20"),
    
    # Dev apps
    ("app-webapp-dev-01", "resource-dev-storage-01", "ReadWrite", "Resource", "2025-09-01"),
    ("app-api-dev-01", "resource-dev-storage-02", "ReadWrite", "Resource", "2025-09-05"),
    
    # Infrastructure
    ("app-terraform", "resource-prod-vm-web-01", "Contributor", "ResourceGroup", "2025-05-10"),
    ("app-terraform", "resource-prod-vm-api-01", "Contributor", "ResourceGroup", "2025-05-10"),
    ("app-ansible", "resource-dev-vm-01", "Contributor", "ResourceGroup", "2025-05-15"),
    ("app-ansible", "resource-dev-vm-02", "Contributor", "ResourceGroup", "2025-05-15"),
    ("app-cicd-pipeline", "resource-prod-vm-web-01", "Contributor", "Resource", "2025-06-01"),
    
    # Security tools
    ("app-vulnerability-scanner", "resource-prod-vm-web-01", "Read", "Subscription", "2025-08-01"),
    ("app-vulnerability-scanner", "resource-prod-vm-api-01", "Read", "Subscription", "2025-08-01"),
    ("app-vulnerability-scanner", "resource-dev-vm-01", "Read", "Subscription", "2025-08-01"),
    
    # Compliance
    ("app-compliance-checker", "resource-prod-keyvault-01", "List", "Subscription", "2025-07-20"),
    ("app-compliance-checker", "resource-admin-keyvault-01", "List", "Subscription", "2025-07-20"),
    
    # TestRunner
    ("app-testrunner-01", "resource-test-vm-01", "ReadWrite", "ResourceGroup", "2025-09-10"),
    ("app-testrunner-01", "resource-test-vm-02", "ReadWrite", "ResourceGroup", "2025-09-10"),
    ("app-testrunner-01", "resource-test-storage-01", "ReadWrite", "ResourceGroup", "2025-09-10"),
]

has_permission_edges_schema = StructType([
    StructField("source_id", StringType(), False),
    StructField("target_id", StringType(), False),
    StructField("permission_type", StringType(), True),
    StructField("scope", StringType(), True),
    StructField("granted_date", StringType(), True),
])
has_permission_edges_df = spark.createDataFrame(has_permission_edges_data, has_permission_edges_schema)
print(f"✅ Created HAS_PERMISSION edges: {has_permission_edges_df.count()} relationships")

# Edge 4: RETRIEVED_CREDENTIAL (User → PAMVault) - 40 edges
# Derive from PAM sessions
retrieved_credential_edges_df = pam_sessions_df.select(
    col("user_id").alias("source_id"),
    col("vault_id").alias("target_id"),
    col("start_time").alias("checkout_time"),
    lit("ServicePrincipal").alias("credential_type"),
    col("duration_minutes").alias("checkout_duration")
)
print(f"✅ Created RETRIEVED_CREDENTIAL edges: {retrieved_credential_edges_df.count()} relationships")

# Edge 5: SESSION_FOR (PAMSession → AzureResource) - 40 edges
session_for_edges_data = [
    # Contractor sessions - PROBLEMATIC
    ("session-001", "resource-dev-vm-01", "2026-01-14 08:00:00", "Standard", False),
    ("session-023", "resource-prod-sql-01", "2026-01-13 10:00:00", "Elevated", True),  # PROBLEM!
    ("session-031", "resource-prod-keyvault-02", "2026-01-12 14:00:00", "Elevated", True),  # PROBLEM!
    ("session-021", "resource-dev-vm-03", "2026-01-14 01:00:00", "Standard", False),
    ("session-022", "resource-test-vm-01", "2026-01-13 02:00:00", "Standard", False),
    ("session-032", "resource-test-vm-02", "2026-01-11 03:00:00", "Standard", False),
    ("session-039", "resource-dev-vm-02", "2026-01-14 12:00:00", "Standard", False),
    ("session-050", "resource-backup-vm-02", "2026-01-14 12:00:00", "Elevated", True), # PROBLEM!
    
    # Regular sessions
    ("session-002", "resource-dev-vm-01", "2026-01-14 06:00:00", "Standard", False),
    ("session-003", "resource-dev-vm-02", "2026-01-14 04:00:00", "Standard", False),
    ("session-004", "resource-dev-vm-03", "2026-01-13 07:00:00", "Standard", False),
    ("session-005", "resource-dev-storage-01", "2026-01-13 05:00:00", "Standard", False),
    ("session-006", "resource-dev-storage-02", "2026-01-12 06:00:00", "Standard", False),
    ("session-007", "resource-prod-vm-web-01", "2026-01-14 03:00:00", "Elevated", True),
    ("session-008", "resource-prod-vm-api-01", "2026-01-13 06:00:00", "Elevated", True),
    ("session-009", "resource-prod-sql-01", "2026-01-14 05:00:00", "Elevated", True),
    ("session-010", "resource-prod-keyvault-01", "2026-01-13 04:00:00", "Elevated", True),
    
    # Admin sessions
    ("session-011", "resource-admin-keyvault-01", "2026-01-14 07:00:00", "Elevated", True),
    ("session-012", "resource-prod-vm-web-01", "2026-01-13 06:00:00", "Elevated", True),
    ("session-013", "resource-prod-keyvault-01", "2026-01-12 07:00:00", "Elevated", True),
    ("session-014", "resource-admin-keyvault-01", "2026-01-14 02:00:00", "Elevated", True),
    ("session-015", "resource-prod-sql-backup", "2026-01-13 03:00:00", "Elevated", True),
    ("session-024", "resource-prod-keyvault-01", "2026-01-11 06:00:00", "Elevated", True),
    ("session-025", "resource-prod-keyvault-02", "2026-01-10 07:00:00", "Elevated", True),
    ("session-033", "resource-admin-keyvault-01", "2026-01-10 04:00:00", "Elevated", True),
    ("session-038", "resource-prod-vm-web-01", "2026-01-14 16:00:00", "Elevated", True),
    ("session-040", "resource-admin-keyvault-01", "2026-01-14 08:00:00", "Elevated", True),
    
    # More sessions
    ("session-016", "resource-dev-vm-01", "2026-01-11 07:00:00", "Standard", False),
    ("session-017", "resource-dev-vm-02", "2026-01-11 05:00:00", "Standard", False),
    ("session-018", "resource-prod-storage-01", "2026-01-12 04:00:00", "Elevated", True),
    ("session-019", "resource-prod-storage-02", "2026-01-12 03:00:00", "Elevated", True),
    ("session-020", "resource-dev-storage-01", "2026-01-10 06:00:00", "Standard", False),
    ("session-026", "resource-prod-vm-api-01", "2026-01-12 05:00:00", "Elevated", True),
    ("session-027", "resource-prod-vm-api-02", "2026-01-11 04:00:00", "Elevated", True),
    ("session-028", "resource-prod-sql-01", "2026-01-10 05:00:00", "Elevated", True),
    ("session-029", "resource-dev-vm-01", "2026-01-09 07:00:00", "Standard", False),
    ("session-030", "resource-dev-vm-02", "2026-01-09 06:00:00", "Standard", False),
    ("session-034", "resource-prod-storage-01", "2026-01-09 05:00:00", "Elevated", True),
    ("session-035", "resource-prod-sql-02", "2026-01-08 06:00:00", "Elevated", True),
    ("session-036", "resource-dev-storage-01", "2026-01-08 07:00:00", "Standard", False),
    ("session-037", "resource-dev-vm-01", "2026-01-15 00:00:00", "Standard", False),
]

session_for_edges_schema = StructType([
    StructField("source_id", StringType(), False),
    StructField("target_id", StringType(), False),
    StructField("session_start", StringType(), True),
    StructField("elevation_level", StringType(), True),
    StructField("approval_required", BooleanType(), True),
])
session_for_edges_df = spark.createDataFrame(session_for_edges_data, session_for_edges_schema)
print(f"✅ Created SESSION_FOR edges: {session_for_edges_df.count()} relationships")

# Edge 6: VAULT_MANAGES (PAMVault → ServicePrincipal) - 12 edges
vault_manages_edges_data = [
    ("vault-prod", "app-datasyncbot", 90, "2026-01-01"),
    ("vault-prod", "app-backupservice", 90, "2025-12-15"),
    ("vault-prod", "app-webapp-prod-01", 90, "2025-11-20"),
    ("vault-prod", "app-webapp-prod-02", 90, "2025-11-20"),
    ("vault-admin", "app-monitoragent", 60, "2025-12-10"),
    ("vault-admin", "app-terraform", 30, "2026-01-05"),
    ("vault-admin", "app-ansible", 30, "2026-01-05"),
    ("vault-contractor", "app-datasyncbot", 90, "2026-01-10"),  # PROBLEM - contractor vault has prod SP!
    ("vault-app", "app-api-prod-01", 90, "2025-11-25"),
    ("vault-app", "app-api-prod-02", 90, "2025-11-25"),
    ("vault-dev", "app-webapp-dev-01", 180, "2025-10-01"),
    ("vault-dev", "app-api-dev-01", 180, "2025-10-01"),
]

vault_manages_edges_schema = StructType([
    StructField("source_id", StringType(), False),
    StructField("target_id", StringType(), False),
    StructField("credential_rotation_days", IntegerType(), True),
    StructField("last_rotation", StringType(), True),
])
vault_manages_edges_df = spark.createDataFrame(vault_manages_edges_data, vault_manages_edges_schema)
print(f"✅ Created VAULT_MANAGES edges: {vault_manages_edges_df.count()} relationships")



print("\n" + "="*60)
print("EDGE DATA SUMMARY")
print("="*60)
print(f"Total Edge DataFrames: 6 types, 187 relationships")
print(f"⚠️  PLANTED ATTACK PATHS:")
print(f"  - Session #023: Contractor → Prod SQL-01")
print(f"  - Session #031: Contractor → Prod KeyVault-02")
print(f"  - DataSyncBot: 8 resource permissions (over-privileged)")
print(f"  - Contractor Vault manages DataSyncBot (misconfiguration)")
print("="*60)


## Section 3: Graph Building
- Define graph ontology
- Build graph with `GraphSpecBuilder`
- Validate graph schema

In [ ]:
# Section 3: Graph Building
# Build custom graph using GraphSpecBuilder fluent API

from sentinel_graph.builders import GraphSpecBuilder

print("="*60)
print("BUILDING CUSTOM GRAPH FROM DATAFRAMES")
print("="*60)

# Build graph using fluent API
builder = (GraphSpecBuilder.start()

    # ========================================================================
    # Node 1: EntraUser (30 users)
    # ========================================================================
    .add_node("EntraUser")
        .from_dataframe(entra_users_df)
        .with_columns("id", "displayName", "department", "userType", "accountEnabled", 
                      key="id", display="displayName")

    # ========================================================================
    # Node 2: AzureResource (25 resources)
    # ========================================================================
    .add_node("AzureResource")
        .from_dataframe(azure_activity_df)
        .with_columns("ResourceId", "ResourceGroup", "ResourceProvider", "ResourceType", "SubscriptionId",
                      key="ResourceId", display="ResourceId")

    # ========================================================================
    # Node 3: ServicePrincipal (15 service principals)
    # ========================================================================
    .add_node("ServicePrincipal")
        .from_dataframe(audit_logs_df)
        .with_columns("AppId", "ServicePrincipalName", "displayName",
                      key="AppId", display="displayName")

    # ========================================================================
    # Node 4: PAMVault (5 vaults)
    # ========================================================================
    .add_node("PAMVault")
        .from_dataframe(pam_vaults_df)
        .with_columns("vault_id", "vault_name", "sensitivity_level", "compliance_tier",
                      key="vault_id", display="vault_name")

    # ========================================================================
    # Node 5: PAMSession (40 sessions)
    # ========================================================================
    .add_node("PAMSession")
        .from_dataframe(pam_sessions_df)
        .with_columns("session_id", "start_time", "duration_minutes", "elevation_reason", "approval_status",
                      key="session_id", display="session_id")

    # ========================================================================
    # Edge 1: ACCESSED (User → Resource) - 45 edges
    # ========================================================================
    .add_edge("ACCESSED")
        .from_dataframe(accessed_edges_df)
        .source(id_column="source_id", node_type="EntraUser")
        .target(id_column="target_id", node_type="AzureResource")
        .with_columns("source_id", "target_id", "operation", "timestamp", "result_status",
                      key="source_id", display="operation")

    # ========================================================================
    # Edge 2: OWNS (User → ServicePrincipal) - 15 edges
    # ========================================================================
    .add_edge("OWNS")
        .from_dataframe(owns_edges_df)
        .source(id_column="source_id", node_type="EntraUser")
        .target(id_column="target_id", node_type="ServicePrincipal")
        .with_columns("source_id", "target_id", "created_date", "permission_level",
                      key="source_id", display="permission_level")

    # ========================================================================
    # Edge 3: HAS_PERMISSION (ServicePrincipal → Resource) - 35 edges
    # ========================================================================
    .add_edge("HAS_PERMISSION")
        .from_dataframe(has_permission_edges_df)
        .source(id_column="source_id", node_type="ServicePrincipal")
        .target(id_column="target_id", node_type="AzureResource")
        .with_columns("source_id", "target_id", "permission_type", "scope", "granted_date",
                      key="source_id", display="permission_type")

    # ========================================================================
    # Edge 4: RETRIEVED_CREDENTIAL (User → PAMVault) - 40 edges
    # ========================================================================
    .add_edge("RETRIEVED_CREDENTIAL")
        .from_dataframe(retrieved_credential_edges_df)
        .source(id_column="source_id", node_type="EntraUser")
        .target(id_column="target_id", node_type="PAMVault")
        .with_columns("source_id", "target_id", "checkout_time", "credential_type", "checkout_duration",
                      key="source_id", display="credential_type")

    # ========================================================================
    # Edge 6: VAULT_MANAGES (PAMVault → ServicePrincipal) - 12 edges
    # ========================================================================
    .add_edge("VAULT_MANAGES")
        .from_dataframe(vault_manages_edges_df)
        .source(id_column="source_id", node_type="PAMVault")
        .target(id_column="target_id", node_type="ServicePrincipal")
        .with_columns("source_id", "target_id", "credential_rotation_days", "last_rotation",
                      key="source_id", display="last_rotation")

    # ========================================================================
    # Edge 7: INITIATED_SESSION (User → PAMSession) - 12 edges
    # ========================================================================
    # Links users to the sessions they initiated - establishes chain of custody
    .add_edge("INITIATED_SESSION")
        .from_dataframe(pam_sessions_df)
        .source(id_column="user_id", node_type="EntraUser")
        .target(id_column="session_id", node_type="PAMSession")
        .with_columns("user_id", "session_id", "start_time", "elevation_reason", "approval_status",
                      key="user_id", display="start_time")

    # ========================================================================
    # Edge 5: SESSION_FOR (PAMSession → AzureResource) - 40 edges
    # ========================================================================
    .add_edge("SESSION_FOR")
        .from_dataframe(session_for_edges_df)
        .source(id_column="source_id", node_type="PAMSession")
        .target(id_column="target_id", node_type="AzureResource")
        .with_columns("source_id", "target_id", "session_start", "elevation_level", "approval_required",
                      key="source_id", display="elevation_level")

).done()

print("✅ Graph specification built successfully!")
print("\n" + "="*60)
print("GRAPH ONTOLOGY SUMMARY")
print("="*60)
print("Nodes: 5 types, 115 total")
print("  - EntraUser:        30 (20 regular, 5 contractors, 5 admins)")
print("  - AzureResource:    25 (3 SQL, 10 VMs, 7 Storage, 5 KeyVaults)")
print("  - ServicePrincipal: 15 (including DataSyncBot)")
print("  - PAMVault:         5 (prod, dev, admin, contractor, app)")
print("  - PAMSession:       40 sessions")
print("\nEdges: 6 types, 187 total")
print("  - ACCESSED:              45 (User → Resource)")
print("  - OWNS:                  15 (User → ServicePrincipal)")
print("  - HAS_PERMISSION:        35 (ServicePrincipal → Resource)")
print("  - RETRIEVED_CREDENTIAL:  40 (User → PAMVault)")
print("  - SESSION_FOR:           40 (PAMSession → Resource)")
print("  - VAULT_MANAGES:         12 (PAMVault → ServicePrincipal)")
print("="*60)
print("\n⚠️  PLANTED ATTACK PATHS:")
print("  1. Alex Chen (Contractor) → Contractor Vault → Session #023 → Prod SQL-01")
print("  2. DataSyncBot (8 resource permissions) - over-privileged")
print("  3. Contractor Vault manages DataSyncBot - misconfiguration")
print("="*60)


Now build the graph

In [ ]:
build_result = builder.build_graph_with_data()

print(f"Status: {build_result.get('status')}")

## Section 3.5: Hunt Over Graph

### ⭐ Explore: Jordan Park's universe

In [ ]:
# My most central admin
query = """
    MATCH (u:EntraUser)-[e1]->(x)-[e2]->(y)
    WHERE u.displayName = 'Jordan Park' 
    RETURN *
"""

result = builder.query(query)
result.show(format="visual")

## ⭐ Explore: Which users could have accessed resources through Service Principals via PAM? 

This is a bit more complex query, where I want to specify details every step of the way.

If user Alice retrieves a Service Principal credential from "Production Vault", and that SP has write access to a production database, then Alice effectively has write access to that database—even though her direct permissions might not show this.

This is exactly the kind of transitive access relationship that graph databases excel at discovering, which would be difficult to query in traditional relational databases.

In [ ]:
# Show all relationship types emanating from PAMVault

query = """
MATCH   (u:EntraUser)-[e2:RETRIEVED_CREDENTIAL]->(v)
        -[e1:VAULT_MANAGES]->(sp:ServicePrincipal)
        -[e3:HAS_PERMISSION]->(r:AzureResource)
WHERE v.vault_name = 'Production Vault' 
RETURN *
"""

result = builder.query(query)
result.show(format="visual")


### Explore: Who's accessing my web servers via PAM sessions?

**Purpose:** Show the expected access path

In [ ]:
query = """
MATCH   (u:EntraUser)-[e1:ACCESSED]->(r:AzureResource),
        (u)-[e4:INITIATED_SESSION]-(ps:PAMSession)-[e3:SESSION_FOR]->(r)
WHERE r.ResourceId in ['resource-prod-vm-web-01', 'resource-prod-vm-web-02']
RETURN *
"""

result_path = builder.query(query)
result_path.show(format="visual")

### ⭐ Hunt For: Unmanaged Privileged Access

**Purpose**: Detect Users who accessed resources WITHOUT a corresponding PAM session - detecting privileged access that bypassed PAM controls.


In [ ]:
from pyspark.sql.functions import col

# First get all PAM session from user to resource
query = """
MATCH   (u:EntraUser)-[e1:RETRIEVED_CREDENTIAL]->(v:PAMVault),
             (u)-[e4:INITIATED_SESSION]-(ps:PAMSession)-[e3:SESSION_FOR]->(r:AzureResource)
RETURN *
"""

result_pam = builder.query(query)
result_pam_df = result_pam.to_dataframe()

# Second, get all accessed resources from user to resource
query = """
MATCH   (u:EntraUser)-[e1:ACCESSED]->(r:AzureResource)
RETURN *
"""

result_accessed = builder.query(query)
result_accessed_df = result_accessed.to_dataframe()

# Extract ResourceId from the string column 'r' 
# The builder.query() returns r as a string, not a struct
filtered_accessed_df = result_accessed_df.join(
    result_pam_df.select(col("r").alias("pam_ResourceId")),
    result_accessed_df["r"] == col("pam_ResourceId"),
    "left_anti"
)

filtered_accessed_df.show(n=5, truncate=False, vertical=True)



### Explore: Selected Service Principal ownership

**Purpose:** Demonstrate the User → OWNS → ServicePrincipal → HAS_PERMISSION → Resource chain

**What This Shows:** Admin Jordan Park owns DataSyncBot, which has excessive permissions (8 resources). This is the over-privileged service principal we planted.

In [ ]:
# Show service principal ownership and permissions chain
query = """
    MATCH (u:EntraUser)-[e1:OWNS]->(sp:ServicePrincipal)-[e2:HAS_PERMISSION]->(r:AzureResource) 
    WHERE sp.displayName = 'DataSyncBot'
    RETURN *
"""

result = builder.query(query)
result.show(format="visual")


### Explore: Users with service principals to production resources

Turns out there's quite a lot

In [ ]:
query = """
MATCH (u:EntraUser)-[e1:OWNS]->(sp:ServicePrincipal)-[e3:HAS_PERMISSION]->(r:AzureResource)
WHERE r.ResourceGroup = 'Production'
RETURN *
"""

result = builder.query(query)
result.show(format="visual")

### Explore: Owners of Service principals to production key vaults

In [ ]:
query = """
MATCH (u:EntraUser)-[e1:OWNS]->(sp:ServicePrincipal)-[e3:HAS_PERMISSION]->(r:AzureResource)
WHERE r.ResourceGroup = 'Production' AND r.ResourceType = 'Key Vault'
RETURN *
"""

result = builder.query(query)
result.show(format="visual")

## Section 4: Algorithm Demonstrations
- **4.1 Centrality:** Find critical assets
- **4.2 K-Hop:** Blast Radius
- **4.3 Reachability** Policy Violation Check
- **4.4 'Blast Radius':** Targeted Path Discovery
- **4.5 Ranked Paths:** Attack Path Analysis


### 1. Centrality - "Find Critical Assets"

**Purpose:** Identify which nodes are most critical based on connectivity

**Narrative Impact:** Security team now knows which assets are most interconnected and therefore highest value targets for attackers. If any of these are compromised, the impact will be significant.

**Visualization:** Bar chart showing top 10 nodes by centrality score

In [ ]:
from sentinel_graph.builders.query_input import CentralityQueryInput
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, TimestampType
from pyspark.sql.functions import from_json, col, schema_of_json, lit
import matplotlib.pyplot as plt
import json

result = builder.centrality(CentralityQueryInput(
    participating_source_node_labels=["EntraUser", "PAMVault", "AzureResource", "ServicePrincipal"],
    participating_target_node_labels=["AzureResource", "ServicePrincipal"],
    participating_edge_labels=["ACCESSED", "SESSION_FOR", "VAULT_MANAGES", "HAS_PERMISSION"],
    is_directional=True,
    threshold=5
))

result_df = result.to_dataframe()

# Get schema from first row
first_row = result_df.first()
json_schema = schema_of_json(lit(first_row.Node))

# Parse JSON column and sort in PySpark
df = (
    result_df
    .withColumn("node", from_json(col("Node"), json_schema))
    .select(
        col("node.NumberOfPaths").alias("centrality"),
        col("node.sys_id").alias("node_id")
    )
    .orderBy(col("centrality").desc())  # ← Sort in PySpark
    .limit(10)  # ← Limit to 10 in PySpark
)

# Convert to pandas (already sorted and limited)
chart_data = df.toPandas()

# Sort and plot
chart_data = chart_data.sort_values('centrality', ascending=False)

plt.figure(figsize=(12, 6))
plt.barh(chart_data['node_id'], chart_data['centrality'])
plt.xlabel('Number of Paths (Centrality Score)')
plt.ylabel('Node ID')
plt.title('Node Centrality Analysis')
plt.tight_layout()
plt.show()


### 2. K-Hop - "Blast Radius"

**Purpose:** What is the scope of potential impact of a compromised contractor account?

**Result:** ⚠️ **Answer: Contractor "Alex Chen" can reach 5 entities in 3 hops, including Production SQL Server (via misconfigured PAM session).** This is a critical finding - a low-privilege contractor should NOT have production access.

**Narrative Impact:** What initially seemed like a low-risk contractor compromise now reveals access to production systems. The K-Hop analysis exposes the hidden attack surface. This triggers deeper investigation into HOW this access exists (next: Reachability).

**Security Value:**
- Understand attack surface from compromised account
- Identify lateral movement opportunities
- Prioritize which compromised accounts have largest blast radius

**Visualization:** Network graph showing hop distances with color coding (hop 1=red, hop 2=orange, hop 3=yellow)

In [ ]:
from sentinel_graph.builders.query_input import K_HopQueryInput

contractor_id = "user-contractor-alex-chen"
result = builder.k_hop(K_HopQueryInput(
    source_property_value=contractor_id,
    max_hop_count=3,
    is_directional=True
))
print(f"\n### K-Hop Analysis Result for Contractor {contractor_id} ###")
result.show(format="visual")

### Jordan Park in 3 hops

How central of a user is Jordan? I showed you two hops before, here's 3. It's quite a lot!

In [ ]:
from sentinel_graph.builders.query_input import K_HopQueryInput

admin_id = "user-admin-jordan-park"
result = builder.k_hop(K_HopQueryInput(
    source_property_value=admin_id,
    max_hop_count=3,
    is_directional=True
))
print(f"\n### K-Hop Analysis Result for Admin {admin_id} ###")
result.show(format="visual")

### 3. Reachability - "Policy Violation Check"

**Purpose:** Can contractors reach production resources? (Should be FALSE per policy)

**Expected Results:**
- **ALERT:** 5 contractors CAN reach production (policy violation)
- Path: Contractors → Contractor Vault → DataSyncBot ServicePrincipal → Production SQL
- Root cause: Misconfigured PAM vault permissions

**Result:** 🚨 **Answer: YES, contractors CAN reach production resources (5 contractors, 5 production resources).** This violates the organization's segregation policy. The paths are:
1. Alex Chen → Contractor Vault → PAM Session #23 → Production SQL-01
2. Sam Rivera → Contractor Vault → PAM Session #31 → Production KeyVault-02

**Narrative Impact:** The K-Hop discovery is now confirmed as a policy violation. Reachability proves contractors have forbidden access paths. Root cause: Contractor Vault is misconfigured with production resource permissions. Immediate remediation required.


In [ ]:
from sentinel_graph.builders.query_input import ReachabilityQueryInput

result = builder.reachability(ReachabilityQueryInput(
    source_property="userType",
    source_property_value="Contractor",
    target_property="ResourceGroup",
    target_property_value="Production",
    max_hop_count=5
))

print("\n### Reachability Analysis Result from Contractors to Production Resources ###")
result.show(format="visual")


### 4. Targeted Path Discovery

**Purpose:** If a contractor account is compromised, what attack paths exist to reach critical production resources??

**Expected Results:**
- 3 distinct paths from contractor to production SQL
- Shortest path: 3 hops
- All paths traverse PAM vault (chokepoint)

**Result:** 📊 **Answer: There are 2 distinct attack paths from Alex Chen to Production SQL-01 (shortest: 3 hops).** All paths go through Contractor Vault - this is a critical chokepoint.

But the good new is there is nothing new!

**Path Details:**
1. Alex → Contractor Vault → PAM Session #23 → Prod SQL-01 (3 hops)
2. Alex → Contractor Vault → DataSyncBot SP → Prod SQL-01 (4 hops, via VAULT_MANAGES + HAS_PERMISSION)

**Narrative Impact:** The blast radius analysis shows multiple redundant attack vectors. Even if we block one path, two others remain. However, ALL paths go through Contractor Vault - if we fix that ONE chokepoint, we eliminate all the access.

In [ ]:
from sentinel_graph.builders.query_input import BlastRadiusQueryInput

contractor_id= "user-contractor-alex-chen"
result = builder.blast_radius(BlastRadiusQueryInput(
    source_property_value=contractor_id,  # Alex Chen
    target_property_value="resource-prod-sql-01",  # Critical resource
    max_hop_count=6,
    is_directional=True
))
print(f"\n### Blast Radius Analysis Result for Contractor {contractor_id} ###")
result.show(format="visual")


### 5. Ranked Paths - "Attack Path Analysis"

**Purpose:** Find and rank the most dangerous attack paths from source to specific targets (remediation prioritization)

**⚠️ Under construction** This scenario is still under construction. It would require me to go back and add a risk score property to every bit of source data. First I should make a targeted experiment for just this.

**MSEM Equivalent:** This IS **Attack Path Analysis** - finds prioritized paths between entities based on risk

**Ranked Paths vs K-Hop:**
- **K-Hop:** "Show me everything reachable" (breadth - discovery)
- **Ranked Paths:** "Show me the TOP paths to critical targets ranked by risk" (depth - analysis)

**Expected Results:**
1. **Path to Production SQL** (risk=9.2, 3 hops) - Immediate fix required
2. **Path to Admin KeyVault** (risk=8.7, 4 hops) - High priority
3. **Path to Dev SQL** (risk=4.1, 2 hops) - Monitor only

**Result:** 🎯 **Answer: The TOP attack path is Alex Chen → Contractor Vault → PAM Session #23 → Production SQL-01 (risk score: 9.2/10).** This is the highest priority remediation target.

**Risk Breakdown for Path #1:**
- User privilege: LOW (contractor) = 2/10
- Credential checkout: APPROVED (legitimate session) = 3/10
- Resource sensitivity: CRITICAL (production database) = 10/10
- Path length: SHORT (3 hops) = 8/10
- **Combined Risk:** 9.2/10

**Remediation Priority:**
1. **Immediate (risk 9-10):** Remove Production SQL access from Contractor Vault
2. **High (risk 7-8.9):** Audit Admin KeyVault permissions, add approval gates
3. **Medium (risk 4-6.9):** Review Dev SQL access (acceptable risk for now)

**Narrative Impact:** After discovering the attack surface (K-Hop), confirming policy violations (Reachability), and measuring impact (Blast Radius), we now have a PRIORITIZED action plan. The security team knows exactly which path to fix first for maximum risk reduction.

**Security Value:**
- Prioritize remediation efforts on highest-risk paths
- Understand HOW attackers would reach critical assets
- Weight by exploitability, privilege required, and asset value


In [ ]:
print(dir(RankedQueryInput))

# TODO: Need to add a risk_score property to edges first

result = builder.ranked(RankedQueryInput(
#    source_property_value=contractor_id,
#    target_node_labels=["AzureResource"],
    rank_property_name="risk_score",  # Edge weight property
    max_paths=10,
    threshold=1
))

# Can't do this :(
#risk_score = (
#    user_privilege_level * 0.4 +
#    resource_sensitivity * 0.4 +
#    session_approval_status * 0.2
#)
